In [1]:
import pandas as pd
import json
from collections import defaultdict
from tqdm import tqdm

In [2]:
json_path = r"D:\zeru\user-wallet-transactions.json"

In [3]:
with open(json_path, "r") as file:
    data = json.load(file)

In [5]:
records = []
for item in tqdm(data):
    try:
        record = {
            "wallet": item["userWallet"],
            "action": item["action"],
            "amount": float(item["actionData"].get("amount", 0)) / 1e6,  # USDC uses 6 decimals
            "asset": item["actionData"].get("assetSymbol", None),
            "timestamp": item["timestamp"]
        }
        records.append(record)
    except Exception as e:
        continue

df = pd.DataFrame(records)

100%|██████████████████████████████████████████████████████████████████████| 100000/100000 [00:00<00:00, 907510.33it/s]


In [6]:
df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s")

In [7]:
wallet_features = []

for wallet, group in tqdm(df.groupby("wallet")):
    features = {
        "wallet": wallet,
        "total_txns": len(group),
        "unique_actions": group["action"].nunique(),
        "total_amount": group["amount"].sum(),
        "total_deposit": group[group["action"] == "deposit"]["amount"].sum(),
        "total_borrow": group[group["action"] == "borrow"]["amount"].sum(),
        "total_repay": group[group["action"] == "repay"]["amount"].sum(),
        "total_redeem": group[group["action"] == "redeemunderlying"]["amount"].sum(),
        "total_liquidations": len(group[group["action"] == "liquidationcall"]),
        "days_active": (group["timestamp"].max() - group["timestamp"].min()).days + 1,
    }

    # Ratios
    features["repay_ratio"] = (
        features["total_repay"] / features["total_borrow"]
        if features["total_borrow"] > 0 else 0
    )
    features["deposit_redeem_ratio"] = (
        features["total_deposit"] / features["total_redeem"]
        if features["total_redeem"] > 0 else 0
    )

    wallet_features.append(features)

wallet_df = pd.DataFrame(wallet_features)
wallet_df.to_csv("wallet_features.csv", index=False)
print("✅ Feature extraction complete. Saved to wallet_features.csv")

100%|█████████████████████████████████████████████████████████████████████████████| 3497/3497 [00:05<00:00, 647.80it/s]


✅ Feature extraction complete. Saved to wallet_features.csv
